# Anthropic-Style Emotion Vector Analysis

This notebook adds the final interpretability pass before the demo: projection/probability correlation, bidirectional steering, same-context displacement analysis, explicit nuisance-control comparison, intensity analysis, and a PCA vector-map visualization of the learned emotion directions.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)


def run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)



def ensure_packages() -> None:
    package_map = {
        "yaml": "pyyaml",
        "pandas": "pandas",
        "numpy": "numpy",
        "matplotlib": "matplotlib",
        "seaborn": "seaborn",
        "umap": "umap-learn",
        "torch": "torch",
        "transformers": "transformers",
        "datasets": "datasets",
        "accelerate": "accelerate",
        "sklearn": "scikit-learn",
        "tqdm": "tqdm",
        "huggingface_hub": "huggingface_hub",
    }
    missing = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing:
        run_command([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Required analysis packages are already available.")



def find_local_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd / "FinalProject"]:
        candidate = candidate.resolve()
        if (candidate / "configs" / "wav2vec.yaml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root locally.")



def repo_status_lines(repo_root: Path) -> list[str]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "status", "--short"],
        text=True,
        capture_output=True,
        check=True,
    )
    return [line for line in result.stdout.splitlines() if line.strip()]



def repo_ahead_behind(repo_root: Path) -> tuple[int, int]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "rev-list", "--left-right", "--count", "HEAD...origin/main"],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return 0, 0
    ahead_str, behind_str = result.stdout.strip().split()
    return int(ahead_str), int(behind_str)



def clone_clean_code_checkout(clean_root: Path) -> Path:
    if clean_root.exists():
        shutil.rmtree(clean_root)
    run_command(["git", "clone", "--depth", "1", REPO_URL, str(clean_root)])
    return clean_root



def prepare_project_and_code_roots() -> tuple[Path, Path]:
    runtime_root = Path("/content") / REPO_NAME
    clean_root = Path("/content") / f"{REPO_NAME}-clean"

    if runtime_root.exists() and (runtime_root / ".git").exists():
        try:
            run_command(["git", "-C", str(runtime_root), "fetch", "origin"])
        except subprocess.CalledProcessError as exc:
            print(f"Fetch failed for existing Colab repo; continuing with local state. Details: {exc}")

        status_lines = repo_status_lines(runtime_root)
        ahead, behind = repo_ahead_behind(runtime_root)

        if not status_lines and ahead == 0:
            if behind > 0:
                try:
                    run_command(["git", "-C", str(runtime_root), "pull", "--ff-only", "origin", "main"])
                    code_root = runtime_root
                except subprocess.CalledProcessError as exc:
                    print(f"Fast-forward pull failed; using a clean code checkout instead. Details: {exc}")
                    code_root = clone_clean_code_checkout(clean_root)
            else:
                code_root = runtime_root
        else:
            print(f"Using a clean code checkout because {runtime_root} has local changes or local commits.")
            for line in status_lines[:10]:
                print(" -", line)
            if ahead or behind:
                print(f"Repo divergence relative to origin/main: ahead={ahead}, behind={behind}")
            code_root = clone_clean_code_checkout(clean_root)

        project_root = runtime_root
    elif runtime_root.exists():
        print(f"Using existing non-git project directory for data/artifacts: {runtime_root}")
        project_root = runtime_root
        code_root = clone_clean_code_checkout(clean_root)
    else:
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(runtime_root)])
        project_root = runtime_root
        code_root = runtime_root

    return project_root, code_root


ensure_packages()

if IS_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT, CODE_ROOT = prepare_project_and_code_roots()
else:
    PROJECT_ROOT = find_local_project_root()
    CODE_ROOT = PROJECT_ROOT

def module_uses_code_root(module_file: str | None, code_root: Path) -> bool:
    if module_file is None:
        return False
    try:
        Path(module_file).resolve().relative_to(code_root.resolve())
        return True
    except ValueError:
        return False


os.chdir(PROJECT_ROOT)
for root in [str(CODE_ROOT), str(PROJECT_ROOT)]:
    while root in sys.path:
        sys.path.remove(root)

sys.path.insert(0, str(CODE_ROOT))
if str(PROJECT_ROOT) != str(CODE_ROOT):
    sys.path.insert(1, str(PROJECT_ROOT))

src_module = sys.modules.get("src")
src_file = getattr(src_module, "__file__", None)
if src_module is not None and not module_uses_code_root(src_file, CODE_ROOT):
    stale_modules = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
    for name in stale_modules:
        del sys.modules[name]

print("Project root:", PROJECT_ROOT)
print("Code root:", CODE_ROOT)
print("Working directory:", Path.cwd().resolve())

In [ ]:
from pathlib import Path

experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_output_dir = local_analysis_dir / 'anthropic_style'
local_output_dir.mkdir(parents=True, exist_ok=True)

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_checkpoint_dir = drive_run_root / 'checkpoint' if drive_run_root is not None else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root is not None else None
drive_output_dir = drive_analysis_dir / 'anthropic_style' if drive_analysis_dir is not None else None

checkpoint_dir = local_checkpoint_dir
if not (checkpoint_dir / 'model_state.pt').exists() and drive_checkpoint_dir is not None and (drive_checkpoint_dir / 'model_state.pt').exists():
    checkpoint_dir = drive_checkpoint_dir

analysis_source_dir = local_analysis_dir
if not (analysis_source_dir / 'embedding_arrays.npz').exists() and drive_analysis_dir is not None and (drive_analysis_dir / 'embedding_arrays.npz').exists():
    analysis_source_dir = drive_analysis_dir

metadata_source_path = analysis_source_dir / 'ravdess_metadata_resolved.csv'
if not metadata_source_path.exists():
    metadata_source_path = PROJECT_ROOT / 'data' / 'metadata' / 'ravdess_metadata.csv'

required_files = [
    checkpoint_dir / 'model_state.pt',
    checkpoint_dir / 'config.json',
    checkpoint_dir / 'label_mapping.json',
    analysis_source_dir / 'embedding_arrays.npz',
    analysis_source_dir / 'embedding_metadata.csv',
    analysis_source_dir / 'embedding_summary.json',
    metadata_source_path,
]
missing = [str(path) for path in required_files if not path.exists()]
assert not missing, 'Missing Anthropic-style analysis inputs: ' + ', '.join(missing)

print('Checkpoint source:', checkpoint_dir)
print('Analysis source:', analysis_source_dir)
print('Metadata source:', metadata_source_path)
print('Output directory:', local_output_dir)
if drive_output_dir is not None:
    print('Drive output directory:', drive_output_dir)


In [ ]:
import pandas as pd
from IPython.display import display

from src.analysis.anthropic_style import build_train_directions_with_controls, evaluate_centering_strategies
from src.analysis.emotion_vectors import load_embedding_artifacts

artifacts = load_embedding_artifacts(analysis_source_dir)
source_metadata = pd.read_csv(metadata_source_path)
merge_columns = [column for column in ['file_name', 'repetition_code', 'repetition'] if column in source_metadata.columns]
metadata = artifacts.metadata.merge(source_metadata[merge_columns].drop_duplicates(subset=['file_name']), on='file_name', how='left')

label_names = list(artifacts.summary['label_names'])
reference_label = 'neutral'
reference_idx = label_names.index(reference_label)
label_ids = artifacts.true_label_ids
split_array = metadata['split'].to_numpy()
train_mask = split_array == 'train'
test_mask = split_array == 'test'

final_layer_idx = int(artifacts.layer_embeddings.shape[1] - 1)
final_layer_embeddings = artifacts.layer_embeddings[:, final_layer_idx]
centered_embeddings, centered_centroids, centered_directions = build_train_directions_with_controls(
    embeddings=final_layer_embeddings,
    metadata=metadata,
    label_ids=label_ids,
    label_names=label_names,
    reference_label=reference_label,
)

centering_df = evaluate_centering_strategies(
    embeddings=final_layer_embeddings,
    label_ids=label_ids,
    split_names=metadata['split'],
    actor_ids=metadata['actor_id'],
    statement_codes=metadata['statement_code'],
    label_names=label_names,
)
centering_df.to_csv(local_output_dir / 'centering_strategy_comparison.csv', index=False)

display(centering_df.round(4))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.barplot(
    data=centering_df[centering_df['split'] == 'test'],
    x='variant',
    y='macro_f1',
)
plt.title('Explicit Nuisance-Control Comparison (Test Macro F1)')
plt.xlabel('Embedding variant')
plt.ylabel('Macro F1')
plt.tight_layout()
plt.savefig(local_output_dir / 'centering_strategy_test_macro_f1.png', dpi=200)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.analysis.anthropic_style import (
    build_projection_probability_frame,
    summarize_projection_probability_correlations,
)

projection_probability_df = build_projection_probability_frame(
    metadata=metadata.loc[test_mask].reset_index(drop=True),
    embeddings=centered_embeddings[test_mask],
    probabilities=artifacts.probabilities[test_mask],
    directions=centered_directions,
    label_names=label_names,
    reference_label=reference_label,
)
projection_corr_df = summarize_projection_probability_correlations(projection_probability_df)
projection_probability_df.to_csv(local_output_dir / 'projection_probability_pairs.csv', index=False)
projection_corr_df.to_csv(local_output_dir / 'projection_probability_correlations.csv', index=False)

display(projection_corr_df.round(4))

fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=False)
for ax, target_label in zip(axes, [name for name in label_names if name != reference_label]):
    plot_df = projection_probability_df[projection_probability_df['target_label'] == target_label]
    sns.regplot(
        data=plot_df,
        x='projection',
        y='target_probability',
        scatter_kws={'alpha': 0.45, 's': 30},
        line_kws={'color': 'black'},
        ax=ax,
    )
    corr_value = projection_corr_df.loc[projection_corr_df['target_label'] == target_label, 'pearson_all'].iloc[0]
    ax.set_title(f'{target_label} (r={corr_value:.2f})')
    ax.set_xlabel('Direction projection')
    ax.set_ylabel('Target probability')
plt.suptitle('Projection vs Output Probability on Held-Out Test Clips', y=1.05)
plt.tight_layout()
plt.savefig(local_output_dir / 'projection_probability_scatter_grid.png', dpi=220)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.analysis.emotion_vectors import build_direction_vectors, compute_class_centroids, evaluate_direction_steering
from src.analysis.extract_embeddings import load_trained_checkpoint

checkpoint = load_trained_checkpoint(checkpoint_dir)
classifier_weight = checkpoint.model.classifier.weight.detach().cpu().numpy()
classifier_bias = checkpoint.model.classifier.bias.detach().cpu().numpy()

raw_train_embeddings = final_layer_embeddings[train_mask]
raw_test_embeddings = final_layer_embeddings[test_mask]
train_label_ids = label_ids[train_mask]
test_label_ids = label_ids[test_mask]
raw_centroids = compute_class_centroids(raw_train_embeddings, train_label_ids, num_labels=len(label_names))
raw_directions = build_direction_vectors(raw_centroids, reference_idx)

steering_df = evaluate_direction_steering(
    embeddings=raw_test_embeddings,
    true_label_ids=test_label_ids,
    direction_vectors=raw_directions,
    classifier_weight=classifier_weight,
    classifier_bias=classifier_bias,
    label_names=label_names,
    target_label_ids=[idx for idx, name in enumerate(label_names) if name != reference_label],
    alphas=[-1.0, -0.5, -0.25, 0.0, 0.25, 0.5, 1.0],
)
steering_df.to_csv(local_output_dir / 'bidirectional_steering_summary.csv', index=False)

display(steering_df.round(4))

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=steering_df,
    x='alpha',
    y='delta_target_prob_all',
    hue='target_label',
    marker='o',
)
plt.axhline(0.0, color='black', linewidth=1, linestyle='--')
plt.title('Bidirectional Steering: Target Probability Shift vs Alpha')
plt.xlabel('Alpha (negative suppresses, positive amplifies)')
plt.ylabel('Delta target probability (all test samples)')
plt.tight_layout()
plt.savefig(local_output_dir / 'bidirectional_steering_curves.png', dpi=220)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.analysis.anthropic_style import (
    build_same_context_displacement_frame,
    summarize_same_context_displacements,
)

test_metadata = metadata.loc[test_mask].reset_index(drop=True)
test_centered_embeddings = centered_embeddings[test_mask]

same_context_df = build_same_context_displacement_frame(
    metadata=test_metadata,
    embeddings=test_centered_embeddings,
    directions=centered_directions,
    label_names=label_names,
    reference_label=reference_label,
)
same_context_summary_df = summarize_same_context_displacements(same_context_df)
same_context_df.to_csv(local_output_dir / 'same_context_displacements.csv', index=False)
same_context_summary_df.to_csv(local_output_dir / 'same_context_displacement_summary.csv', index=False)

display(same_context_summary_df.round(4))

plt.figure(figsize=(9, 5))
sns.barplot(data=same_context_summary_df, x='true_label', y='match_rate')
plt.ylim(0, 1.0)
plt.title('Same-Speaker, Same-Sentence, Same-Repetition Direction Match Rate')
plt.xlabel('Emotion label')
plt.ylabel('Matched expected direction')
plt.tight_layout()
plt.savefig(local_output_dir / 'same_context_match_rate.png', dpi=220)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.analysis.anthropic_style import (
    build_intensity_projection_frame,
    build_paired_intensity_delta_frame,
    summarize_intensity_projections,
    summarize_paired_intensity_deltas,
)

non_neutral_mask = metadata['final_label'] != reference_label
intensity_metadata = metadata.loc[non_neutral_mask].reset_index(drop=True)
intensity_embeddings = centered_embeddings[non_neutral_mask]

intensity_projection_df = build_intensity_projection_frame(
    metadata=intensity_metadata,
    embeddings=intensity_embeddings,
    directions=centered_directions,
    label_names=label_names,
    reference_label=reference_label,
)
intensity_summary_df = summarize_intensity_projections(intensity_projection_df)
paired_intensity_df = build_paired_intensity_delta_frame(intensity_projection_df)
paired_intensity_summary_df = summarize_paired_intensity_deltas(paired_intensity_df)

intensity_projection_df.to_csv(local_output_dir / 'intensity_projection_values.csv', index=False)
intensity_summary_df.to_csv(local_output_dir / 'intensity_projection_summary.csv', index=False)
paired_intensity_df.to_csv(local_output_dir / 'paired_intensity_projection_deltas.csv', index=False)
paired_intensity_summary_df.to_csv(local_output_dir / 'paired_intensity_projection_summary.csv', index=False)

display(intensity_summary_df.round(4))
display(paired_intensity_summary_df.round(4))

plt.figure(figsize=(10, 5))
sns.boxplot(data=intensity_projection_df, x='true_label', y='expected_projection', hue='intensity')
plt.title('Expected Direction Projection by Emotion and Intensity')
plt.xlabel('Emotion label')
plt.ylabel('Projection onto expected direction')
plt.tight_layout()
plt.savefig(local_output_dir / 'intensity_projection_boxplot.png', dpi=220)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
pca.fit(centered_embeddings[train_mask])
test_coords = pca.transform(centered_embeddings[test_mask])
direction_coords = centered_directions @ pca.components_.T

vector_plot_df = pd.DataFrame(
    {
        'pca_1': test_coords[:, 0],
        'pca_2': test_coords[:, 1],
        'label': metadata.loc[test_mask, 'final_label'].to_list(),
    }
)
vector_plot_df.to_csv(local_output_dir / 'emotion_vector_pca_points.csv', index=False)

plt.figure(figsize=(10, 8))
sns.scatterplot(data=vector_plot_df, x='pca_1', y='pca_2', hue='label', alpha=0.45, s=40)
for label_name, coords in zip(label_names, direction_coords):
    if label_name == reference_label:
        continue
    plt.arrow(0, 0, coords[0], coords[1], color='black', width=0.01, head_width=0.12, length_includes_head=True)
    plt.text(coords[0] * 1.08, coords[1] * 1.08, f'{label_name} - {reference_label}', fontsize=10)
plt.axhline(0, color='gray', linewidth=1, linestyle='--')
plt.axvline(0, color='gray', linewidth=1, linestyle='--')
plt.title('Emotion Directions Visualized in PCA Space')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.tight_layout()
plt.savefig(local_output_dir / 'emotion_direction_pca_vectors.png', dpi=240)
plt.show()


In [ ]:
print('Anthropic-style analysis directory:', local_output_dir)
for path in sorted(local_output_dir.iterdir()):
    print(path.name)


In [ ]:
if not IS_COLAB:
    print('Google Drive sync is intended for Colab runtimes. Skipping on local execution.')
elif drive_output_dir is None:
    print('Drive output directory is not configured.')
else:
    import shutil

    if drive_output_dir.exists():
        shutil.rmtree(drive_output_dir)
    drive_output_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_output_dir)

    print('Backed up Anthropic-style analysis artifacts to Google Drive:')
    print(' -', drive_output_dir)
    print('\nDrive output contents:')
    for path in sorted(drive_output_dir.iterdir()):
        kind = 'dir' if path.is_dir() else 'file'
        print(f' [{kind}] {path.name}')
